In [1]:
# Neste modulo de classificação vou fazer uma abordagem diferente devido na ultima atividade ter me perdido e passado horas e horas para fazer a atividade devido a um erro de arrendodamento e logaritimo em dados com uma 'calda'. Pretendo dar uma revisada no modulo de regressão linear posteriormente.
# Dei uma olhada nas atividades e vou dar uma olhada nas video aulas enquanto faço as anotações.

# Neste modulo sera abordado um modelo de predição de saida de clientes de uma empresa de telefonia.
# Neste caso sera uma classificação binaria, onde a previsão é se o cliente vai cancelar ou não. sendo 1 certeza de cancelar.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

df = pd.read_csv(data)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Padronizando as informações das colunas e nomes em colunas.
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')
df.head()

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.dtypes
# observamos que possui uma coluna 'totalcharges' que possui numeros mas está caracterizada como string.

customerid              str
gender                  str
seniorcitizen         int64
partner                 str
dependents              str
tenure                int64
phoneservice            str
multiplelines           str
internetservice         str
onlinesecurity          str
onlinebackup            str
deviceprotection        str
techsupport             str
streamingtv             str
streamingmovies         str
contract                str
paperlessbilling        str
paymentmethod           str
monthlycharges      float64
totalcharges            str
churn                   str
dtype: object

In [5]:
# Transformando com pandas os valores em numeros e qualquer erro que houver ignorar se não for numerico.
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')

# Como possui clientes que entraram nesse mes ainda, não possui valores lançados em 'totalcharges' por isso simplificaremos esse valor para 0
df.totalcharges = df.totalcharges.fillna(0)

In [6]:
# Transformando sim em 1 e não em 0. Para podermos usar no target do nosso modelo.
df.churn = (df.churn == 'Yes').astype(int)

In [7]:
from sklearn.model_selection import train_test_split
# Montando o campo de treino, validação e teste do modelo.

df_full_train,df_test = train_test_split(df, test_size=0.2, random_state=1)
len(df_full_train), len(df_test)

(5634, 1409)

In [8]:
df_train,df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)
len(df_train), len(df_val) , len(df_test)

(4225, 1409, 1409)

In [9]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [10]:
y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

In [11]:
del df_train['churn']
del df_val['churn']
del df_test['churn']

In [12]:
# EDA EXPLORATORY DATA ANALYSIS
df_full_train = df_full_train.reset_index(drop=True)

In [15]:
df_full_train.churn.value_counts(normalize=True) # normaliza transforma em ;porcentagem;
# O valores do total
# vemos a proporção de zeros que são pessoas que não sairam e 1 pessoa que sairam

churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

In [16]:
# churn rate
global_churn_rate = df_full_train.churn.mean()
# O valor em media\porcentagem dos valores 1 em relação a 0. No caso seria 26.99% dos assinantes que sairam. Parecido com a função de normaliza em value_counts
round(global_churn_rate, 2)

np.float64(0.27)

In [18]:
# Separando as variaveis das colunas para ser usadas nas features:
numerical = ['tenure','monthlycharges','totalcharges']
categorical = ['gender', 'seniorcitizen', 'partner', 'dependents'
       , 'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod']

In [21]:
# Selecionando o sets que criamos separadamente:
df_full_train[numerical]

tenure              73
monthlycharges    1494
totalcharges      5291
dtype: int64

In [28]:
global_churn = df_full_train.churn.mean()
df_full_train[categorical].nunique()
# Verificando os valores unicos vemos que a maioria é binaria(2 podemos converter para 0 e 1 mais facilmente) e colunas com 3 itens unicos e 4 itens unicos.

gender              2
seniorcitizen       2
partner             2
dependents          2
phoneservice        2
multiplelines       3
internetservice     3
onlinesecurity      3
onlinebackup        3
deviceprotection    3
techsupport         3
streamingtv         3
streamingmovies     3
contract            3
paperlessbilling    2
paymentmethod       4
dtype: int64

In [23]:
# Vamos diferenciar os individuos primeiramente entre masculinos e femininos.
churn_female = df_full_train[df_full_train.gender == 'female'].churn.mean()
churn_male = df_full_train[df_full_train.gender == 'male'].churn.mean()
# churn_female = 0.27682403433476394
# churn_male   = 0.2632135306553911
# Como vemos a taxa são quase iguais e vemos que a coluna genero não influencia tanto no churn rate.


In [24]:
df_full_train.partner.value_counts()

partner
No     2932
Yes    2702
Name: count, dtype: int64

In [25]:
# Aqui vemos uma diferença significamente maior de saida de dos individuos sem parceiros daqueles que possuem parceiro.
churn_partner = df_full_train[df_full_train.partner == 'yes'].churn.mean()
churn_no_partner = df_full_train[df_full_train.partner == 'no'].churn.mean()
# churn_partner     = 0.20503330866025166
#churn_no_partner  = 0.3298090040927694


In [ ]:
# Calculamos: saidas de um grupo de individuos \saidas globais
# churn_no_partner / global_churn
# 1.2216593879412643
# churn_partner / global_churn
# 0.7594724924338315

# Lemos o risk ratior de uma forma mais facil como:

# risk = 1 significa para o grupo que é provavel de sair como todos os outros
# risk > 1 significa para o grupo que é mais provavel de sair
# risk < 1 significa para o grupo que é menos provavel de sair


# Mas possui uma forma de realizar essa verificação em cada grupo especifico usando sql query:


SELECT
    gender,
    AVG(churn),
    AVG(churn) - global_churn AS diff,
    AVG(churn) / global_churn AS risk


FROM
    data


GROUP BY
    gender;

In [29]:

from IPython.display import display
# Criando um display para a função:
for c in categorical:
    print(c)# Mostrar a coluna
    df_group = df_full_train.groupby(c).churn.agg(['mean', 'count']) # Nessa coluna associamos a uma variavel para mostrar a media e a contangem de valores unicos.
    df_group['diff'] = df_group['mean'] - global_churn # Nessa coluna mostramos a diferença bruta
    df_group['risk'] = df_group['mean'] / global_churn # Nessa coluna mostramos o risk ratio
    display(df_group)
    print()
    print()

gender


,mean,count,diff,risk
gender,,,,
Female,0.276824,2796,0.006856,1.025396
Male,0.263214,2838,-0.006755,0.974980




seniorcitizen


,mean,count,diff,risk
seniorcitizen,,,,
0,0.242270,4722,-0.027698,0.897403
1,0.413377,912,0.143409,1.531208




partner


...

,mean,count,diff,risk
contract,,,,
Month-to-month,0.431701,3104,0.161733,1.599082
One year,0.120573,1186,-0.149395,0.446621
Two year,0.028274,1344,-0.241694,0.104730




paperlessbilling


,mean,count,diff,risk
paperlessbilling,,,,
No,0.172071,2313,-0.097897,0.637375
Yes,0.338151,3321,0.068183,1.252560




paymentmethod


,mean,count,diff,risk
paymentmethod,,,,
Bank transfer (automatic),0.168171,1219,-0.101797,0.622928
Credit card (automatic),0.164339,1217,-0.105630,0.608733
Electronic check,0.455890,1893,0.185922,1.688682
Mailed check,0.193870,1305,-0.076098,0.718121
